# Named Entity Recognition System using Transformers

This project uses a pretrained Transformer model from Hugging Face to detect named entities from text.

The system identifies entities such as:
- Person
- Organization
- Location
- Date and Time (Optional)
- Miscellaneous entities

In [23]:
# ============================================================
# STEP 1: IMPORT LIBRARIES AND CHECK DEVICE
# ============================================================

# Core libraries
import os
import re
import json
import torch
import spacy
import pandas as pd
from datetime import datetime
from typing import List, Dict

# Hugging Face libraries
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Notebook display
from IPython.display import display, HTML

# Check whether GPU is available
DEVICE = 0 if torch.cuda.is_available() else -1

if torch.cuda.is_available():
    print("GPU Available:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Running on CPU.")

GPU Available: NVIDIA GeForce GTX 1050 Ti with Max-Q Design


In [24]:
# ============================================================
# STEP 2: DEFINE PROJECT CONFIGURATION
# ============================================================

SMALL_MODEL_NAME = "dslim/bert-base-NER"
LARGE_MODEL_NAME = "dbmdz/bert-large-cased-finetuned-conll03-english"

OUTPUT_CSV_PATH = "outputs/extracted_entities.csv"

LABEL_MAPPING = {
    "PER": "PERSON",
    "ORG": "ORGANIZATION",
    "LOC": "LOCATION",
    "MISC": "MISCELLANEOUS",
    "DATE": "DATE",
    "TIME": "TIME"
}

ENTITY_COLORS = {
    "PERSON": "#ff6b6b",
    "ORGANIZATION": "#4dabf7",
    "LOCATION": "#51cf66",
    "DATE": "#ffd43b",
    "TIME": "#b197fc",
    "MISCELLANEOUS": "#ffa94d"
}

In [25]:
# ============================================================
# STEP 3: LOAD SMALL OR LARGE TRANSFORMER NER MODEL
# ============================================================

def load_ner_pipeline(model_size="large"):
    """
    Loads a pretrained Hugging Face Transformer model for Named Entity Recognition.

    Parameters:
        model_size (str): 
            'small' -> memory-friendly BERT base model.
            'large' -> higher-capacity BERT large model.

    Returns:
        transformers.Pipeline: Ready-to-use NER pipeline.
    """

    if model_size.lower() == "large":
        model_name = LARGE_MODEL_NAME
    else:
        model_name = SMALL_MODEL_NAME

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name)

    ner_pipeline = pipeline(
        task="ner",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="simple",
        device=DEVICE
    )

    print(f"Model loaded successfully: {model_name}")
    print(f"Device: {'GPU' if DEVICE == 0 else 'CPU'}")

    return ner_pipeline


ner_model = load_ner_pipeline(model_size="large")

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model loaded successfully: dbmdz/bert-large-cased-finetuned-conll03-english
Device: GPU


In [26]:
# ============================================================
# STEP 4: EXTRACT DATE AND TIME USING REGEX
# ============================================================

def is_overlapping(new_entity, existing_entities):
    """
    Checks whether a new entity overlaps with an already extracted entity.

    This prevents duplicate time matches such as:
    - 10:30 AM
    - 30 AM
    """

    new_start = new_entity["start"]
    new_end = new_entity["end"]

    for entity in existing_entities:
        existing_start = entity["start"]
        existing_end = entity["end"]

        if new_start < existing_end and new_end > existing_start:
            return True

    return False


def extract_date_time_entities(text):
    """
    Extracts date and time entities using regex patterns.

    This is added because most CoNLL-03 NER models mainly detect:
    PERSON, ORGANIZATION, LOCATION, and MISC.

    Date/time detection is handled as an extra project feature.
    This updated version also removes duplicate overlapping matches.
    """

    date_patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
        r"\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}\b",
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2},?\s+\d{4}\b",
        r"\b\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4}\b",
        r"\b\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4}\b"
    ]

    time_patterns = [
        r"\b\d{1,2}:\d{2}\s?(?:AM|PM|am|pm)?\b",
        r"(?<!:)\b\d{1,2}\s?(?:AM|PM|am|pm)\b"
    ]

    entities = []

    for pattern in date_patterns:
        for match in re.finditer(pattern, text):
            entity = {
                "entity": match.group(),
                "label": "DATE",
                "score": 1.0,
                "start": match.start(),
                "end": match.end(),
                "source": "Regex"
            }

            if not is_overlapping(entity, entities):
                entities.append(entity)

    for pattern in time_patterns:
        for match in re.finditer(pattern, text):
            entity = {
                "entity": match.group(),
                "label": "TIME",
                "score": 1.0,
                "start": match.start(),
                "end": match.end(),
                "source": "Regex"
            }

            if not is_overlapping(entity, entities):
                entities.append(entity)

    return entities

In [27]:
# ============================================================
# STEP 5: MAIN ENTITY EXTRACTION FUNCTION
# ============================================================

def extract_entities(text, ner_pipeline, confidence_threshold=0.70, include_date_time=True):
    """
    Extracts named entities from user text using Transformer NER model.

    Parameters:
        text (str): Input sentence or paragraph.
        ner_pipeline: Hugging Face NER pipeline.
        confidence_threshold (float): Minimum confidence score.
        include_date_time (bool): Adds regex-based DATE and TIME extraction.

    Returns:
        pd.DataFrame: Structured entity extraction results.
    """

    transformer_results = ner_pipeline(text)
    final_entities = []

    for item in transformer_results:
        raw_label = item.get("entity_group", "")
        mapped_label = LABEL_MAPPING.get(raw_label, raw_label)
        score = round(float(item.get("score", 0)), 4)

        if score >= confidence_threshold:
            final_entities.append({
                "entity": item.get("word", ""),
                "label": mapped_label,
                "score": score,
                "start": item.get("start", None),
                "end": item.get("end", None),
                "source": "Transformer"
            })

    if include_date_time:
        final_entities.extend(extract_date_time_entities(text))

    df = pd.DataFrame(final_entities)

    if not df.empty:
        df = df.sort_values(by=["start", "end"]).reset_index(drop=True)

    return df

In [28]:
# ============================================================
# STEP 6: TEST THE NER SYSTEM ON SAMPLE TEXT
# ============================================================

sample_text = """
Elon Musk founded Tesla in California in 2003. 
Apple opened a new office in London on 12 March 2024 at 10:30 AM.
"""

entities_df = extract_entities(
    text=sample_text,
    ner_pipeline=ner_model,
    confidence_threshold=0.70
)

entities_df

,entity,label,score,start,end,source
0,Elon Musk,PERSON,0.9975,1,10,Transformer
1,Tesla,ORGANIZATION,0.9868,19,24,Transformer
2,California,LOCATION,0.9997,28,38,Transformer
3,Apple,ORGANIZATION,0.9985,49,54,Transformer
4,London,LOCATION,0.9993,78,84,Transformer
5,12 March 2024,DATE,1.0000,88,101,Regex
6,10:30 AM,TIME,1.0000,105,113,Regex


In [29]:
# ============================================================
# STEP 7: DISPLAY ENTITIES IN CLEAN PROFESSIONAL FORMAT
# ============================================================

def display_entities_table(entities_df):
    """
    Displays extracted entities in a clean tabular format.
    """

    if entities_df.empty:
        print("No entities found.")
        return

    display(entities_df[["entity", "label", "score", "source"]])


display_entities_table(entities_df)

,entity,label,score,source
0,Elon Musk,PERSON,0.9975,Transformer
1,Tesla,ORGANIZATION,0.9868,Transformer
2,California,LOCATION,0.9997,Transformer
3,Apple,ORGANIZATION,0.9985,Transformer
4,London,LOCATION,0.9993,Transformer
5,12 March 2024,DATE,1.0000,Regex
6,10:30 AM,TIME,1.0000,Regex


In [30]:
# ============================================================
# STEP 8: HIGHLIGHT ENTITIES IN COLORED TEXT
# ============================================================

def highlight_entities(text, entities_df):
    """
    Highlights detected entities inside the original text using colored HTML spans.
    """

    if entities_df.empty:
        display(HTML(f"<p>{text}</p>"))
        return

    highlighted_text = ""
    last_index = 0

    valid_entities = entities_df.dropna(subset=["start", "end"]).sort_values(by="start")

    for _, row in valid_entities.iterrows():
        start = int(row["start"])
        end = int(row["end"])
        label = row["label"]
        entity = text[start:end]
        color = ENTITY_COLORS.get(label, "#adb5bd")

        highlighted_text += text[last_index:start]
        highlighted_text += (
            f"<span style='background-color:{color}; color:black; padding:3px 6px; "
            f"border-radius:6px; font-weight:bold;'>{entity} ({label})</span>"
        )
        last_index = end

    highlighted_text += text[last_index:]

    display(HTML(f"<div style='font-size:16px; line-height:1.8'>{highlighted_text}</div>"))


highlight_entities(sample_text, entities_df)

In [31]:
# ============================================================
# STEP 9: SAVE EXTRACTED ENTITIES TO CSV
# ============================================================

def save_entities_to_csv(entities_df, text, file_path=OUTPUT_CSV_PATH):
    """
    Saves extracted entities into a CSV file.

    Each record includes:
    - original text
    - entity value
    - entity label
    - confidence score
    - extraction source
    - timestamp
    """

    if entities_df.empty:
        print("No entities available to save.")
        return

    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    save_df = entities_df.copy()
    save_df["input_text"] = text
    save_df["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if os.path.exists(file_path):
        existing_df = pd.read_csv(file_path)
        final_df = pd.concat([existing_df, save_df], ignore_index=True)
    else:
        final_df = save_df

    final_df.to_csv(file_path, index=False)

    print(f"Entities saved successfully to: {file_path}")


save_entities_to_csv(entities_df, sample_text)

Entities saved successfully to: outputs/extracted_entities.csv


In [32]:
# ============================================================
# STEP 10: CUSTOM USER INPUT FUNCTION
# ============================================================

def run_custom_ner(text, model_size="large", confidence_threshold=0.70):
    """
    Runs the complete NER pipeline on custom user text.

    This function:
    - loads selected model
    - extracts entities
    - displays table
    - highlights entities
    - saves results to CSV
    """

    ner_pipeline = load_ner_pipeline(model_size=model_size)

    results_df = extract_entities(
        text=text,
        ner_pipeline=ner_pipeline,
        confidence_threshold=confidence_threshold,
        include_date_time=True
    )

    display_entities_table(results_df)
    highlight_entities(text, results_df)
    save_entities_to_csv(results_df, text)

    return results_df


custom_text = "Barack Obama visited Microsoft headquarters in Seattle on 15 April 2025 at 9 AM."

custom_results = run_custom_ner(
    text=custom_text,
    model_size="large",
    confidence_threshold=0.70
)

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model loaded successfully: dbmdz/bert-large-cased-finetuned-conll03-english
Device: GPU


,entity,label,score,source
0,Barack Obama,PERSON,0.9982,Transformer
1,Microsoft,ORGANIZATION,0.9988,Transformer
2,Seattle,LOCATION,0.9990,Transformer
3,15 April 2025,DATE,1.0000,Regex
4,9 AM,TIME,1.0000,Regex


Entities saved successfully to: outputs/extracted_entities.csv


In [33]:
# ============================================================
# STEP 11: INTERACTIVE NER SYSTEM
# ============================================================

def interactive_ner_system(ner_pipeline, confidence_threshold=0.70):
    """
    Runs an interactive command-line NER system inside Jupyter Notebook.
    Type 'exit' to stop the system.
    """

    print("NER System Started")
    print("Type 'exit' to stop.\n")

    while True:
        user_input = input("Enter text: ")

        if user_input.lower().strip() == "exit":
            print("NER System Closed.")
            break

        results_df = extract_entities(
            text=user_input,
            ner_pipeline=ner_pipeline,
            confidence_threshold=confidence_threshold,
            include_date_time=True
        )

        display_entities_table(results_df)
        highlight_entities(user_input, results_df)
        save_entities_to_csv(results_df, user_input)


interactive_ner_system(ner_model, confidence_threshold=0.70)

NER System Started
Type 'exit' to stop.

NER System Closed.


In [34]:
# ============================================================
# STEP 12: COMPARE TRANSFORMER NER WITH SPACY NER
# ============================================================

import spacy

def load_spacy_model():
    """
    Loads spaCy English model for NER comparison.
    """

    try:
        return spacy.load("en_core_web_sm")
    except OSError:
        print("spaCy model not found. Run: python -m spacy download en_core_web_sm")
        return None


def compare_spacy_transformer(text, transformer_pipeline, confidence_threshold=0.70):
    """
    Compares entities extracted by Transformer model and spaCy model.
    """

    transformer_df = extract_entities(
        text=text,
        ner_pipeline=transformer_pipeline,
        confidence_threshold=confidence_threshold,
        include_date_time=True
    )

    spacy_nlp = load_spacy_model()
    spacy_entities = []

    if spacy_nlp:
        doc = spacy_nlp(text)

        for ent in doc.ents:
            spacy_entities.append({
                "entity": ent.text,
                "label": ent.label_,
                "score": None,
                "start": ent.start_char,
                "end": ent.end_char,
                "source": "spaCy"
            })

    spacy_df = pd.DataFrame(spacy_entities)

    comparison_df = pd.concat([transformer_df, spacy_df], ignore_index=True)

    return comparison_df


comparison_text = "Google was founded by Larry Page and Sergey Brin in California on 4 September 1998."

comparison_results = compare_spacy_transformer(
    text=comparison_text,
    transformer_pipeline=ner_model,
    confidence_threshold=0.70
)

comparison_results

c:\MiniForge\envs\intern_env\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
C:\Users\Saif Ullah\AppData\Local\Temp\ipykernel_21676\1642336775.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparison_df = pd.concat([transformer_df, spacy_df], ignore_index=True)


,entity,label,score,start,end,source
0,Google,ORGANIZATION,0.9984,0,6,Transformer
1,Larry Page,PERSON,0.9991,22,32,Transformer
2,Sergey Brin,PERSON,0.9963,37,48,Transformer
3,California,LOCATION,0.9995,52,62,Transformer
4,4 September 1998,DATE,1.0000,66,82,Regex
5,Google,ORG,NaN,0,6,spaCy
6,Larry Page,PERSON,NaN,22,32,spaCy
7,Sergey Brin,PERSON,NaN,37,48,spaCy
8,California,GPE,NaN,52,62,spaCy
9,September 1998,DATE,NaN,68,82,spaCy


In [35]:
# ============================================================
# STEP 13: BATCH NER PROCESSING FOR MULTIPLE SENTENCES
# ============================================================

def batch_extract_entities(text_list, ner_pipeline, confidence_threshold=0.70):
    """
    Processes multiple text inputs and returns all extracted entities in one DataFrame.
    """

    all_results = []

    for text in text_list:
        df = extract_entities(
            text=text,
            ner_pipeline=ner_pipeline,
            confidence_threshold=confidence_threshold,
            include_date_time=True
        )

        if not df.empty:
            df["input_text"] = text
            all_results.append(df)

    if all_results:
        return pd.concat(all_results, ignore_index=True)

    return pd.DataFrame()


texts = [
    "Elon Musk founded SpaceX in California.",
    "Microsoft opened an office in London on 10 May 2023.",
    "Imran Khan gave a speech in Islamabad at 8 PM."
]

batch_results = batch_extract_entities(texts, ner_model, confidence_threshold=0.70)

batch_results

,entity,label,score,start,end,source,input_text
0,Elon Musk,PERSON,0.9984,0,9,Transformer,Elon Musk founded SpaceX in California.
1,SpaceX,ORGANIZATION,0.9988,18,24,Transformer,Elon Musk founded SpaceX in California.
2,California,LOCATION,0.9996,28,38,Transformer,Elon Musk founded SpaceX in California.
3,Microsoft,ORGANIZATION,0.9995,0,9,Transformer,Microsoft opened an office in London on 10 May...
4,London,LOCATION,0.9995,30,36,Transformer,Microsoft opened an office in London on 10 May...
5,10 May 2023,DATE,1.0000,40,51,Regex,Microsoft opened an office in London on 10 May...
6,Imran Khan,PERSON,0.9992,0,10,Transformer,Imran Khan gave a speech in Islamabad at 8 PM.
7,Islamabad,LOCATION,0.9997,28,37,Transformer,Imran Khan gave a speech in Islamabad at 8 PM.
8,8 PM,TIME,1.0000,41,45,Regex,Imran Khan gave a speech in Islamabad at 8 PM.


In [36]:
# ============================================================
# STEP 14: FINAL PROJECT TEST FUNCTION
# ============================================================

def final_ner_test():
    """
    Final testing function for the complete NER system.
    """

    text = input("Enter text for NER analysis: ")
    model_choice = input("Choose model size: small or large: ").lower().strip()

    if model_choice not in ["small", "large"]:
        model_choice = "small"

    confidence = float(input("Enter confidence threshold, example 0.70: "))

    results = run_custom_ner(
        text=text,
        model_size=model_choice,
        confidence_threshold=confidence
    )

    return results


# Run this when needed:
# final_ner_test()